# Week 7: Mixture of Experts (MoE)

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week07-moe.html

**How to use:** open this notebook in the course codespace (folder `notebooks/`) and run the cells from the top (Shift+Enter, or Run All). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need your key as a Codespaces secret or in the `.env` file. For Ollama examples first run `bash scripts/start-ollama.sh` in the terminal. Model-training examples are much faster on a GPU: open this notebook in Google Colab and choose Runtime → Change runtime type → T4 GPU (in Colab add API keys with the 🔑 Secrets panel).

In [ ]:
# Imports that some of the lesson's code fragments assume
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## Step 1: The Expert Network — a feedforward sub-model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# A single Expert = a 2-layer feedforward network (same as a transformer FFN)
class Expert(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)   # expand
        self.fc2 = nn.Linear(d_ff, d_model)   # project back

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [num_tokens, d_model]
        return self.fc2(F.relu(self.fc1(x)))

# Try it out
expert = Expert(d_model=512, d_ff=2048)
tokens = torch.randn(4, 512)   # batch of 4 tokens
output = expert(tokens)
print(f"Input:  {tokens.shape}")   # [4, 512]
print(f"Output: {output.shape}")   # [4, 512] — same shape!

## Step 2: The Router — deciding which experts process each token

In [ ]:
class TopKRouter(nn.Module):
    """Routes each token to the top-k most relevant experts."""
    def __init__(self, d_model: int, num_experts: int, top_k: int = 2):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts
        # Learned: maps token embedding → score per expert
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x: torch.Tensor):
        # x: [num_tokens, d_model]
        # Step A: raw routing scores for every expert
        logits = self.gate(x)                          # [num_tokens, num_experts]
        # Step B: softmax → probability distribution
        probs = F.softmax(logits, dim=-1)              # [num_tokens, num_experts]
        # Step C: pick the top-k experts per token
        top_k_probs, top_k_idx = torch.topk(probs, self.top_k, dim=-1)
        # top_k_probs: [num_tokens, k] — how much weight per expert
        # top_k_idx:   [num_tokens, k] — which expert indices
        # Step D: renormalize so the k weights sum to 1
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        return top_k_probs, top_k_idx, probs   # probs used for load-balancing loss

# Example: 4 tokens, 8 experts, select top-2
router = TopKRouter(d_model=512, num_experts=8, top_k=2)
tokens = torch.randn(4, 512)
weights, indices, all_probs = router(tokens)

print("Expert indices per token:", indices)
# e.g. tensor([[3, 6], [1, 7], [0, 2], [5, 3]])
print("Expert weights per token: ", weights.round(decimals=2))
# e.g. tensor([[0.62, 0.38], [0.71, 0.29], ...])

## Step 3: The MoE Layer — combining router + experts

In [ ]:
class MoELayer(nn.Module):
    """
    Sparse Mixture of Experts Layer.
    Replaces a single FFN in a Transformer with N expert FFNs.
    Only k out of N experts activate per token (k << N).
    """
    def __init__(self, d_model: int, d_ff: int,
                 num_experts: int = 8, top_k: int = 2):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts
        # N independent expert networks
        self.experts = nn.ModuleList([
            Expert(d_model, d_ff) for _ in range(num_experts)
        ])
        # Router selects which experts process each token
        self.router = TopKRouter(d_model, num_experts, top_k)

    def forward(self, x: torch.Tensor):
        # x: [batch, seq_len, d_model]
        B, S, D = x.shape
        # Flatten batch and sequence dims — treat all tokens equally
        x_flat = x.reshape(-1, D)                   # [B*S, d_model]
        N = x_flat.shape[0]                         # total tokens

        # Get routing decisions
        weights, indices, all_probs = self.router(x_flat)
        # weights: [N, top_k] — how much each expert contributes
        # indices: [N, top_k] — which expert to use

        output = torch.zeros_like(x_flat)           # [N, d_model]

        # For each expert slot (0 to top_k-1)
        for k in range(self.top_k):
            expert_ids  = indices[:, k]             # [N] which expert for slot k
            slot_weight = weights[:, k:k+1]         # [N, 1] contribution weight

            # Group tokens by their assigned expert and run each expert once
            for e in range(self.num_experts):
                mask = (expert_ids == e)            # which tokens go to expert e
                if not mask.any():
                    continue                        # skip unused experts (saves compute!)
                expert_out = self.experts[e](x_flat[mask])   # [n_e, D]
                output[mask] += slot_weight[mask] * expert_out  # weighted add

        return output.reshape(B, S, D), all_probs   # restore original shape

# Test the full MoE layer
moe = MoELayer(d_model=512, d_ff=2048, num_experts=8, top_k=2)
x = torch.randn(2, 10, 512)                         # batch=2, seq_len=10
out, probs = moe(x)
print(f"Input:  {x.shape}")                         # torch.Size([2, 10, 512])
print(f"Output: {out.shape}")                       # torch.Size([2, 10, 512]) ✓

# Count parameters: 8 experts × (512×2048 + 2048 + 2048×512 + 512) = ~16.8M expert params
# (plus 8×512 = 4,096 router weights). Only 2/8 experts activate per token = 25% active
total = sum(p.numel() for p in moe.parameters())
active = total * (moe.top_k / moe.num_experts)
print(f"Total params: {total:,} | Active per token: {int(active):,}")

## Step 4: Load Balancing Loss — preventing expert collapse

In [ ]:
def auxiliary_load_balancing_loss(
    router_probs: torch.Tensor,   # [N, num_experts] — softmax output
    expert_indices: torch.Tensor, # [N, top_k] — selected expert IDs
    num_experts: int
) -> torch.Tensor:
    """
    Switch Transformer / Mixtral style auxiliary load-balancing loss.
    Encourages all experts to receive roughly equal token counts.

    Loss = num_experts * sum_i(f_i * P_i)
      f_i = fraction of tokens routed to expert i
      P_i = mean router probability for expert i
    Both terms should be ~1/num_experts for uniform distribution.
    """
    N = router_probs.shape[0]

    # f_i: fraction of tokens dispatched to each expert
    one_hot = F.one_hot(expert_indices, num_experts).float()   # [N, k, E]
    tokens_per_expert = one_hot.sum(dim=[0, 1]) / (N * expert_indices.shape[1])
    # Shape: [num_experts] — ideally all equal to 1/num_experts

    # P_i: mean router probability assigned to each expert
    mean_probs = router_probs.mean(dim=0)   # Shape: [num_experts] — ideally all equal to 1/num_experts

    # Dot product × num_experts: 1.0 when perfectly balanced
    loss = num_experts * (tokens_per_expert * mean_probs).sum()
    return loss

# ------ How to use it in your training loop ------
moe = MoELayer(d_model=512, d_ff=2048, num_experts=8, top_k=2)
x = torch.randn(2, 10, 512)

out, all_probs = moe(x)
_, indices, _ = moe.router(x.reshape(-1, 512))

# Main task loss (e.g. cross-entropy for language modelling)
task_loss = torch.tensor(2.5)   # placeholder

# Add a small load-balancing penalty (coefficient α = 0.01)
balance_loss = auxiliary_load_balancing_loss(all_probs, indices, num_experts=8)
total_loss = task_loss + 0.01 * balance_loss

print(f"Task loss:    {task_loss.item():.4f}")
print(f"Balance loss: {balance_loss.item():.4f}")   # ideally close to 1.0
print(f"Total loss:   {total_loss.item():.4f}")

## Step 5: Using Mixtral / DeepSeek via HuggingFace Transformers

In [ ]:
# Install: pip install transformers accelerate bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# ── Option A: Mixtral 8x7B (open weights, Apache-2.0) ──────────────────────
model_name = "mistralai/Mixtral-8x7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load with 4-bit quantization: reduces VRAM from ~90 GB → ~24 GB
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",        # spread across available GPUs/CPU
    torch_dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),   # requires bitsandbytes
)

# Prepare a chat-formatted prompt
messages = [
    {"role": "user", "content": "Explain Mixture of Experts in three sentences."}
]
input_ids = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to(model.device)

# Generate a response
with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(
    output_ids[0][input_ids.shape[1]:],
    skip_special_tokens=True
)
print(response)

# ── Option B: Inspect which experts were activated ─────────────────────────
# Access the router logits for interpretability
outputs = model(input_ids, output_router_logits=True)
router_logits = outputs.router_logits   # tuple of tensors per MoE layer
for layer_idx, logits in enumerate(router_logits):
    top2 = logits.topk(2, dim=-1).indices   # [seq, 2]
    print(f"Layer {layer_idx}: top-2 experts used → {top2[0].tolist()}")

## Run in your browser

**A Mixture-of-Experts layer with top-2 routing (NumPy)**

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
d_model, d_ff, n_experts, top_k, n_tokens = 8, 16, 4, 2, 6

W_gate = rng.normal(0, 0.5, (d_model, n_experts))
experts = [(rng.normal(0, 0.3, (d_model, d_ff)), rng.normal(0, 0.3, (d_ff, d_model))) for _ in range(n_experts)]
x = rng.normal(size=(n_tokens, d_model))

logits = x @ W_gate
probs = np.exp(logits - logits.max(axis=1, keepdims=True)); probs /= probs.sum(axis=1, keepdims=True)
chosen = np.argsort(probs, axis=1)[:, ::-1][:, :top_k]

y = np.zeros_like(x)
load = np.zeros(n_experts, dtype=int)
for t in range(n_tokens):
    g = probs[t, chosen[t]]; g = g / g.sum()               # renormalise over the chosen experts
    for gate, e in zip(g, chosen[t]):
        W1, W2 = experts[e]
        y[t] += gate * (np.maximum(0, x[t] @ W1) @ W2)      # FFN expert with ReLU
        load[e] += 1
    print(f"token {t}: experts {chosen[t].tolist()} with weights {np.round(g, 2).tolist()}")

params_per_expert = d_model * d_ff + d_ff * d_model
print("\nload per expert:", load)
print(f"total expert parameters: {n_experts * params_per_expert}, used per token: {top_k * params_per_expert} "
      f"({top_k / n_experts:.0%})")

## Test code for the course environment

**The core routing step on its own: softmax → top-k → renormalize**

In [ ]:
# 1. The core routing step on its own: softmax → top-k → renormalize
import torch, torch.nn.functional as F
torch.manual_seed(0)
num_experts, top_k = 8, 2
gate = torch.nn.Linear(512, num_experts, bias=False)
x = torch.randn(4, 512)                       # 4 tokens
probs = F.softmax(gate(x), dim=-1)
w, idx = torch.topk(probs, top_k, dim=-1)
w = w / w.sum(dim=-1, keepdim=True)
print("experts chosen per token:\n", idx)
print("renormalized weights (rows sum to 1):\n", w.round(decimals=2), w.sum(-1))

**Show that an MoE layer preserves shape and uses only k/N of its expert parameters**

In [ ]:
# 2. Show that an MoE layer preserves shape and uses only k/N of its expert parameters
import torch, torch.nn as nn, torch.nn.functional as F

class Expert(nn.Module):
    def __init__(s, d, f): super().__init__(); s.fc1, s.fc2 = nn.Linear(d, f), nn.Linear(f, d)
    def forward(s, x): return s.fc2(F.relu(s.fc1(x)))

class MoE(nn.Module):
    def __init__(s, d=64, f=256, E=8, k=2):
        super().__init__(); s.k, s.E = k, E
        s.experts = nn.ModuleList(Expert(d, f) for _ in range(E)); s.gate = nn.Linear(d, E, bias=False)
    def forward(s, x):
        B, S, D = x.shape; xf = x.reshape(-1, D)
        p = F.softmax(s.gate(xf), -1); w, idx = torch.topk(p, s.k, -1); w = w / w.sum(-1, keepdim=True)
        out = torch.zeros_like(xf)
        for slot in range(s.k):
            for e in range(s.E):
                m = idx[:, slot] == e
                if m.any(): out[m] += w[m, slot:slot+1] * s.experts[e](xf[m])
        return out.reshape(B, S, D), p, idx

moe = MoE(); y, p, idx = moe(torch.randn(2, 10, 64))
total = sum(q.numel() for q in moe.experts.parameters())
print("output shape:", tuple(y.shape), "| expert params:", total, "| active per token:", total * moe.k // moe.E)

**Load-balancing loss: balanced vs. collapsed routing**

In [ ]:
# 3. Load-balancing loss: balanced vs. collapsed routing
import torch, torch.nn.functional as F
def aux_loss(probs, idx, E):
    f = F.one_hot(idx, E).float().sum(dim=[0, 1]) / (probs.shape[0] * idx.shape[1])
    P = probs.mean(0)
    return E * (f * P).sum()
E, N = 8, 1000
uniform_p = torch.full((N, E), 1 / E); uniform_idx = torch.stack([torch.arange(N) % E, (torch.arange(N) + 1) % E], 1)
collapsed_p = F.one_hot(torch.zeros(N, dtype=torch.long), E).float() * 0.9 + 0.1 / E
collapsed_idx = torch.zeros(N, 2, dtype=torch.long)
print("balanced  :", aux_loss(uniform_p, uniform_idx, E).item())    # ≈ 1.0
print("collapsed :", aux_loss(collapsed_p, collapsed_idx, E).item()) # >> 1.0

**Watch the auxiliary loss fix collapse during a toy training loop**

In [ ]:
# 4. Watch the auxiliary loss fix collapse during a toy training loop
import torch, torch.nn.functional as F
torch.manual_seed(1)
E, k, d = 8, 2, 32
gate = torch.nn.Linear(d, E, bias=False)
with torch.no_grad(): gate.weight[0] += 3.0          # bias the router toward expert 0 (collapse)
opt = torch.optim.Adam(gate.parameters(), lr=0.05)
for step in range(201):
    x = torch.randn(256, d)
    p = F.softmax(gate(x), -1); _, idx = torch.topk(p, k, -1)
    f = F.one_hot(idx, E).float().sum([0, 1]) / (256 * k)
    loss = E * (f * p.mean(0)).sum()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step:3d}  L_aux={loss.item():.3f}  tokens/expert={(f*100).round().int().tolist()}%")